In [2]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

seed = 1234
np.random.seed(seed)  


In [5]:
# read data
data = pd.read_csv('diabetes+130-us+hospitals+for+years+1999-2008/diabetic_data.csv')
# save original dataset
original_data = data.copy()

features_to_drop = ['weight', 'payer_code', 'medical_specialty']
data = data.drop(features_to_drop, axis=1)
# data['medical_specialty'] = data['medical_specialty'].replace('?', 'Missing')
data = data.replace('?', np.nan)

#dropping rows with missing race for now
data = data.dropna(subset='race')
data['diag_1'] = pd.to_numeric(data['diag_1'], errors='coerce')

data['diag_1'] = data['diag_1'].fillna('Missing')

data['diag_2'] = pd.to_numeric(data['diag_2'], errors='coerce')

data['diag_2'] = data['diag_2'].fillna('Missing')

data['diag_3'] = pd.to_numeric(data['diag_3'], errors='coerce')

data['diag_3'] = data['diag_3'].fillna('Missing')
data.drop_duplicates(['patient_nbr'], keep = 'first', inplace = True)

print(len(data))

intervalDict = {'[0-10)' : 5,
'[10-20)' : 15,
'[20-30)' : 25, 
'[30-40)' : 35, 
'[40-50)' : 45, 
'[50-60)' : 55,
'[60-70)' : 65, 
'[70-80)' : 75,
'[80-90)' : 85,
'[90-100)' : 95}

data['age'] = data['age'].apply(lambda x : intervalDict[x])

# 1 = Emergency/Urgent, 5 = Not Available/NULL/Not Mapped, 3 = Elective, 4 = Newborn, 7 = Trauma Center
data['admission_type_id'] = data['admission_type_id'].apply(lambda x : 1 if int(x) in [1, 2] 
                                                            else (5 if int(x) in [5, 6, 8] 
                                                                  else int(x) ))
# 1 = Referral, 2 = Transfer, 3 = Nan, 4 = Delivery, etc. 
data['admission_source_id'] = data['admission_source_id'].apply(lambda x : 1 if int(x) in [2, 3] 
                                                                else (2 if int(x) in [4, 5, 6, 10, 22, 25] 
                                                                      else (3 if int(x) in [9, 15, 17, 20, 21] 
                                                                            else (4 if int(x) in [11, 13, 14] 
                                                                                  else int(x)))))

# discharge disposition categorization
data['discharge_disposition_id'] = data['discharge_disposition_id'].apply(
    lambda x: 1 if int(x) in [6, 8, 9, 13]
    else (2 if int(x) in [3, 4, 5, 14, 22, 23, 24]  # discharged to facility
    else (10 if int(x) in [12, 15, 16, 17]          # hospice
    else (11 if int(x) in [19, 20, 21]              # expired
    else (18 if int(x) in [25, 26]                  # psychiatric
      else int(x)))))
)

# remove patients that died or hospice related
data = data[~data.discharge_disposition_id.isin([11,13,14,19,20,21])]



# diagnostic categorization
def categorize_diagnosis(diag):

    if diag == 'Missing':
        return 'Missing'

    try:
        diag = float(diag)
    except:
        return 'Other'

    if 390 <= diag <= 459 or diag == 785:
        return 'Circulatory'
    elif 460 <= diag <= 519 or diag == 786:
        return 'Respiratory'
    elif 520 <= diag <= 579 or diag == 787:
        return 'Digestive'
    elif 250 <= diag < 251:
        return 'Diabetes'
    elif 800 <= diag <= 999:
        return 'Injury'
    elif 710 <= diag <= 739:
        return 'Musculoskeletal'
    elif 580 <= diag <= 629 or diag == 788:
        return 'Genitourinary'
    elif 140 <= diag <= 239:
        return 'Neoplasms'
    else:
        return 'Other'
    
data['diag_1_cat'] = data['diag_1'].apply(categorize_diagnosis)
data['diag_2_cat'] = data['diag_2'].apply(categorize_diagnosis)
data['diag_3_cat'] = data['diag_3'].apply(categorize_diagnosis)
data = data.drop(['diag_1', 'diag_2', 'diag_3'], axis=1) # dropping the original columns bc new ones were created

# response variable
data['readmitted'] = data['readmitted'].apply(lambda x: 1 if x == '<30' else 0)
data['change'] = data['change'].apply(lambda x : 1 if x == 'Ch'
                                                 else 0)


data['diabetesMed'] = data['diabetesMed'].apply(lambda x : 0 if x == 'No'
                                                else 1)

for col in ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride", "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide", "examide", "citoglipton", "insulin", "glyburide-metformin", "glipizide-metformin", "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"]:
    data[col] = data[col].apply(lambda x : 1 if x == 'Up' 
            else ( -1 if x == 'Down'                                                          
            else ( 0 if x == 'Steady'
            else  -2)))


data['max_glu_serum'] = data['max_glu_serum'].apply(lambda x : 200 if x == '>200' 
                                                            else ( 300 if x == '>300'                                                          
                                                            else ( 100 if x == 'Norm'
                                                            else  0)))

data['A1Cresult'] = data['A1Cresult'].apply(lambda x : 7 if x == '>7' 
                                                         else (8 if  x == '>8'                                                        
                                                         else ( 5 if x == 'Norm'
                                                         else  0)))

data = pd.get_dummies(data, columns=['admission_type_id','admission_source_id', 'gender', 'race'])

data.head()
# print(len(data))

69668


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,5,5,25,1,1,41,...,No,No,No,No,No,No,No,-1,-1,NO
1,149190,55629189,Caucasian,Female,15,1,1,7,3,59,...,No,Up,No,No,No,No,No,1,1,>30
2,64410,86047875,AfricanAmerican,Female,25,1,1,7,2,11,...,No,No,No,No,No,No,No,-1,1,NO
3,500364,82442376,Caucasian,Male,35,1,1,7,2,44,...,No,Up,No,No,No,No,No,1,1,NO
4,16680,42519267,Caucasian,Male,45,1,1,7,1,51,...,No,Steady,No,No,No,No,No,1,1,NO


In [4]:
# split test/train

# separate features and target
X = data.drop("readmitted", axis=1)
y = data["readmitted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

print(X.shape)
print(X.columns)
# verify the shape
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(69668, 46)
Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
       'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
       'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton',
       'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed'],
      dtype='object')
(55734, 46)
(13934, 46)
(55734,)
(13934,)
